# Qwen3-1.7B LoRA Fine-Tuning for 6GB VRAM

Adapted for RTX 3050 6GB class GPUs.

In [ ]:
pip_install = '''
pip install transformers datasets accelerate peft trl bitsandbytes sentencepiece evaluate tensorboard wandb
'''
print(pip_install)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig


In [ ]:
MODEL_NAME='Qwen/Qwen3-1.7B'
MAX_SEQ_LENGTH=256
OUTPUT_DIR='./qwen3-rbi-qa-lora'
RUN_NAME='qwen3-rbi-qa'


In [ ]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=compute_dtype,
)

model.config.use_cache = False


In [ ]:
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
)


In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=2,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=5,
    save_steps=100,
    eval_steps=100,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_length=MAX_SEQ_LENGTH,
    report_to='wandb',
    run_name=RUN_NAME,
)
print(sft_config)
